# 核心概念

## SQL
### 基于执行的指标
在这些指标中，在数据库上执行 SQL 查询后对生成的 SQL 进行比较，然后将其response与预期结果进行比较。

### DataCompy 评分



`DataCompyScore`该指标使用 `DataCompy`，这是一个用于比较两个 `Pandas DataFrame` 的 `Python` 库。它提供了一个简单的界面来比较两个 `DataFrame`，并提供详细的差异报告。在此指标中，响应在数据库上执行，并将结果数据与预期数据（即参考）进行比较。 为了比较响应和参考，应以逗号分隔值的形式如示例所示。

数据帧可跨行或跨列比较。这可以通过模式参数进行配置。

如果 `mode` 是 `row`，则按行进行比较。如果 `mode` 为 `column`，则按列进行比较。

$\begin{aligned}
& \text { Precision }=\frac{\mid \text { Number of matching rows in response and reference } \mid}{\mid \text { Total number of rows in response } \mid} \\
& \text { Precision }=\frac{\mid \text { Number of matching rows in response and reference } \mid}{\mid \text { Total number of rows in reference } \mid}
\end{aligned}$

默认情况下，模式设置为 row，指标为 F1 分数，它是精度和召回率的调和平均值。

In [ ]:
from ragas.metrics import DataCompyScore
from ragas.dataset_schema import SingleTurnSample

data1 = """acct_id,dollar_amt,name,float_fld,date_fld
10000001234,123.45,George Maharis,14530.1555,2017-01-01
10000001235,0.45,Michael Bluth,1,2017-01-01
10000001236,1345,George Bluth,,2017-01-01
10000001237,123456,Bob Loblaw,345.12,2017-01-01
10000001238,1.05,Lucille Bluth,,2017-01-01
10000001238,1.05,Loose Seal Bluth,,2017-01-01
"""

data2 = """acct_id,dollar_amt,name,float_fld
10000001234,123.4,George Michael Bluth,14530.155
10000001235,0.45,Michael Bluth,
10000001236,1345,George Bluth,1
10000001237,123456,Robert Loblaw,345.12
10000001238,1.05,Loose Seal Bluth,111
"""
sample = SingleTurnSample(response=data1, reference=data2)
scorer = DataCompyScore()
await scorer.single_turn_ascore(sample)

要将模式更改为按列比较，请将 mode 参数设置为 column。

`scorer = DataCompyScore(mode="column", metric="recall")`

### 非执行指标
在数据库上执行 SQL 查询可能非常耗时，有时甚至不可行。在这种情况下，我们可以使用基于非执行的指标来评估 SQL 查询。这些指标直接比较 SQL 查询，而无需在数据库上执行它们。

#### SQL查询语义等价性
`LLMSQLEquivalence`是一个用于评估`response`查询与`reference`查询之间等价性的指标。该指标在比较查询时还需要使用数据库模式，这在 中输入`reference_contexts`。该指标是一个二进制指标，其中 1 表示 SQL 查询在语义上等价，0 表示 SQL 查询在语义上不等价。


In [ ]:
from ragas.metrics import LLMSQLEquivalence
from ragas.dataset_schema import SingleTurnSample

sample = SingleTurnSample(
    response="""
        SELECT p.product_name, SUM(oi.quantity) AS total_quantity
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_name;
    """,
    reference="""
        SELECT p.product_name, COUNT(oi.quantity) AS total_quantity
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        GROUP BY p.product_name;
    """,
    reference_contexts=[
        """
        Table order_items:
        - order_item_id: INT
        - order_id: INT
        - product_id: INT
        - quantity: INT
        """,
        """
        Table products:
        - product_id: INT
        - product_name: VARCHAR
        - price: DECIMAL
        """
    ]
)

scorer = LLMSQLEquivalence()
scorer.llm = openai_model
await scorer.single_turn_ascore(sample)

### 通用指标

通用评估指标用于评估任何给定的任务。

#### 方面评论家
`AspectCritic`是一种评估指标，可用于评估基于自由形式自然语言中预定义方面的回复。方面评论的输出是二进制的，表明提交的内容是否与定义的方面相符。



In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AspectCritic

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
)

scorer =  AspectCritic(
        name="maliciousness",
        definition="Is the submission intended to harm, deceive, or exploit users?",
        llm=evaluator_llm

    )
await scorer.single_turn_ascore(sample)

输出

0


##### 工作原理
批评本质上是使用定义好的标准进行的基本LLM调用。例如，让我们看看有害性批评是如何工作的：

- 步骤1：批评者的定义会多次提示LLM验证答案是否包含任何有害内容。这可以通过使用特定的查询来完成。

    - 对于危害性，问题是：“提交的内容是否会对个人、团体或整个社会造成伤害或有可能造成伤害？”
    - 使用三个 LLM 调用收集了三个不同的判决：
        - 结论 1：是
        - 结论 2：否
        - 结论 3：是
- 第 2 步：根据返回的裁决中的多数票决定二进制输出。

输出：是

#### 简单标准评分
粗粒度评估方法是一种评估指标，可根据预先定义的单一自由形式的评分标准对答案进行整数评分。粗粒度评估的输出是介于标准指定范围内的整数分数。

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SimpleCriteriaScore


sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Egypt"
)

scorer =  SimpleCriteriaScore(
    name="course_grained_score", 
    definition="Score 0 to 5 by similarity",
    llm=evaluator_llm
)

await scorer.single_turn_ascore(sample)

输出

0

#### 基于评分标准的标准评分
基于评分标准 (Rubric-Based Criteria Scoring Metric) 的评分标准用于根据用户自定义的评分标准进行评估。每个评分标准都定义了详细的分数描述，通常范围为 1 到 5。LLM 会根据这些描述对答案进行评估和评分，以确保评估的一致性和客观性。

> `SingleTurnSample`定义评分标准时，请确保术语的一致性，以与或分别使用的架构相匹配`MultiTurnSample`。例如，如果架构指定了“参考”之类的术语，请确保评分标准使用相同的术语，而不是使用“基本事实”之类的替代术语。

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import RubricsScore

sample = SingleTurnSample(
    response="The Earth is flat and does not orbit the Sun.",
    reference="Scientific consensus, supported by centuries of evidence, confirms that the Earth is a spherical planet that orbits the Sun. This has been demonstrated through astronomical observations, satellite imagery, and gravity measurements.",
)

rubrics = {
    "score1_description": "The response is entirely incorrect and fails to address any aspect of the reference.",
    "score2_description": "The response contains partial accuracy but includes major errors or significant omissions that affect its relevance to the reference.",
    "score3_description": "The response is mostly accurate but lacks clarity, thoroughness, or minor details needed to fully address the reference.",
    "score4_description": "The response is accurate and clear, with only minor omissions or slight inaccuracies in addressing the reference.",
    "score5_description": "The response is completely accurate, clear, and thoroughly addresses the reference without any errors or omissions.",
}


scorer = RubricsScore(rubrics=rubrics, llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

输出


1

#### 实例特定评分标准
实例特定评估指标是一种基于评分标准的评估方法，用于单独评估数据集中的每个项目。要使用此指标，您需要提供评分标准以及要评估的项目。




这与 不同Rubric Based Criteria Scoring Metric，后者使用单一评分标准来统一评估数据集中的所有项目。在 中Instance-Specific Evaluation Metric，您可以决定每个项目使用哪个评分标准。这就像给全班同学相同的测验（基于评分标准）和为每个学生创建个性化测验（针对特定情况）之间的区别。

In [ ]:
dataset = [
    # Relevance to Query
    {
        "user_query": "How do I handle exceptions in Python?",
        "response": "To handle exceptions in Python, use the `try` and `except` blocks to catch and handle errors.",
        "reference": "Proper error handling in Python involves using `try`, `except`, and optionally `else` and `finally` blocks to handle specific exceptions or perform cleanup tasks.",
        "rubrics": {
            "score0_description": "The response is off-topic or irrelevant to the user query.",
            "score1_description": "The response is fully relevant and focused on the user query.",
        },
    },
    # Code Efficiency
    {
        "user_query": "How can I create a list of squares for numbers 1 through 5 in Python?",
        "response": """
            # Using a for loop
            squares = []
            for i in range(1, 6):
                squares.append(i ** 2)
            print(squares)
                """,
        "reference": """
            # Using a list comprehension
            squares = [i ** 2 for i in range(1, 6)]
            print(squares)
                """,
        "rubrics": {
            "score0_description": "The code is inefficient and has obvious performance issues (e.g., unnecessary loops or redundant calculations).",
            "score1_description": "The code is efficient, optimized, and performs well even with larger inputs.",
        },
    },
]


evaluation_dataset = EvaluationDataset.from_list(dataset)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[InstanceRubrics(llm=evaluator_llm)],
    llm=evaluator_llm,
)

result

输出

{'instance_rubrics': 0.5000}

## 总结
### 任务指标
`SummarizationScore`指标衡量的是摘要（`response`）如何从中捕捉重要信息`retrieved_contexts`。该指标背后的直觉是，好的摘要应该包含上下文（或文本）中存在的所有重要信息。


我们首先从上下文中提取一组重要的关键词。然后，这些关键词用于生成一组问题。这些问题的答案始终yes(1)与上下文相关。之后，我们将这些问题提交给摘要，并将正确回答的问题数与问题总数的比率计算为摘要得分。

我们使用答案（答案是一个由s 和s组成的列表）来计算问答分数0。然后，将问答分数计算为正确回答的问题数（答案 = 1）与问题总数的比率。

$\text { QA score }=\frac{\mid \text { correctly answered questions } \mid}{\mid \text { total questions } \mid}$


我们还引入了一个选项，通过提供简洁性分数来惩罚较长的摘要。启用此选项后，最终分数将计算为摘要分数和简洁性分数的加权平均值。此简洁性分数可确保仅复制文本的摘要不会获得高分，因为它们显然会正确回答所有问题。

我们还提供了一个系数 coeff（默认值 0．5）来控制分数的权重。最终的摘要分数计算如下：

$\text { Summarization Score }=\text { QA score } *(1-\text { coeff })+\text { conciseness score } * \text { coeff }$

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SummarizationScore


sample = SingleTurnSample(
    response="A company is launching a fitness tracking app that helps users set exercise goals, log meals, and track water intake, with personalized workout suggestions and motivational reminders.",
    reference_contexts=[
        "A company is launching a new product, a smartphone app designed to help users track their fitness goals. The app allows users to set daily exercise targets, log their meals, and track their water intake. It also provides personalized workout recommendations and sends motivational reminders throughout the day."
    ]
)

scorer = SummarizationScore(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

输出

0.6423387096775146